# ATLAS: bounded Gemma 4 E4B runtime probe (v4, kernel numerics + chat template)

v3 proved that the source-level tile guard makes the quantized E4B checkpoint run on
a Tesla T4, but the 32 generated tokens were degenerate. v4 answers two questions and
nothing else:

1. **Kernel numerics** — the tile-16 Triton attention kernel is compared against a
   float32 SDPA reference for head_dim 256/512, prefill and decode. If it disagrees,
   the guard is unusable and every downstream experiment is void.
2. **Chat template** — v1-v3 fed raw strings to an instruction-tuned model; v4 repeats
   the same prompts through the tokenizer's chat template (with a manual Gemma-format
   fallback) to test whether the degenerate output is a prompting artifact.

The numeric check runs before the model load and aborts the probe if it fails.
Everything stays bounded: one attempt, FP16, tensor parallelism 1, 2,048-token
context, subprocess with external timeout. This probe establishes runtime
compatibility and output character only; it never measures agent repair performance.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)
START = time.monotonic()
(WORK / 'install_runtime.py').write_text('import glob\nimport importlib\nimport logging\nimport os\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n# Configure environment variables for offline vLLM serving and LiteLLM routing\nos.environ[\'TRANSFORMERS_NO_TF\'] = \'1\'\nos.environ[\'TRANSFORMERS_NO_FLAX\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nos.environ[\'VLLM_NO_USAGE_STATS\'] = \'1\'\nos.environ[\'VLLM_WORKER_MULTIPROC_METHOD\'] = \'spawn\'\nos.environ[\'VLLM_ENGINE_READY_TIMEOUT_S\'] = \'1200\'\nos.environ[\'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS\'] = \'1\'\nos.environ[\'NCCL_DEBUG\'] = \'WARN\'\nos.environ[\'LITELLM_LOCAL_MODEL_COST_MAP\'] = \'True\'\nos.environ[\'LITELLM_TELEMETRY\'] = \'False\'\nos.environ[\'LITELLM_LOG\'] = \'WARNING\'\nos.environ[\'OPENAI_API_KEY\'] = os.environ.get(\'OPENAI_API_KEY\', \'EMPTY\')\nos.environ[\'LOCAL_API_KEY\'] = os.environ.get(\'LOCAL_API_KEY\', \'EMPTY\')\nos.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\nos.environ[\'OTEL_SDK_DISABLED\'] = \'true\'\nos.environ[\'OTEL_PYTHON_DISABLED\'] = \'true\'\n\n\ndef configure_logging() -> None:\n    """Silence verbose third-party loggers."""\n    for name in (\n        \'LiteLLM\',\n        \'LiteLLM Router\',\n        \'LiteLLM Proxy\',\n        \'litellm\',\n        \'httpx\',\n        \'httpcore\',\n        \'openai\',\n        \'urllib3\',\n        \'opentelemetry\',\n        \'opentelemetry.context\',\n    ):\n        lg = logging.getLogger(name)\n        lg.setLevel(logging.WARNING)\n        if \'litellm\' in name.lower():\n            lg.propagate = False\n            lg.handlers.clear()\n\n\nconfigure_logging()\n\nWHEELHOUSE_DIR = Path(\'/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse\')\n\n# Remove broken cutlass .pth hooks if present\nfor pth_pattern in (\n    \'/usr/local/lib/python*/dist-packages/*cutlass*.pth\',\n    \'/usr/local/lib/python*/site-packages/*cutlass*.pth\',\n):\n    for pth in glob.glob(pth_pattern):\n        try:\n            os.unlink(pth)\n        except OSError:\n            pass\n\n# Restore PEP 440 \'+cu128\' wheel filenames stripped by Kaggle dataset uploads\ntmp_whl = Path(\'/tmp/wheelhouse\')\ntmp_whl.mkdir(parents=True, exist_ok=True)\nfor w in WHEELHOUSE_DIR.glob(\'*.whl\'):\n    if \'cutlass\' in w.name.lower():\n        continue\n    target_name = (\n        w.name.replace(\'cu128\', \'+cu128\')\n        if (\'cu128\' in w.name and \'+\' not in w.name)\n        else w.name\n    )\n    target = tmp_whl / target_name\n    if not target.exists():\n        try:\n            os.symlink(w, target)\n        except OSError:\n            shutil.copyfile(w, target)\n\nwheels = sorted(str(w) for w in tmp_whl.glob(\'*.whl\'))\nprint(f\'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...\')\nsubprocess.run(\n    [sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'--no-deps\', \'--force-reinstall\', *wheels],\n    check=True,\n)\nimportlib.invalidate_caches()\nconfigure_logging()\nprint(\'Wheelhouse installation complete.\')', encoding='utf-8')
with (WORK / 'install.log').open('w') as log:
    try:
        process = subprocess.run([sys.executable, str(WORK / 'install_runtime.py')], stdout=log, stderr=subprocess.STDOUT, timeout=420)
        installation = {'status': 'passed' if process.returncode == 0 else 'failed', 'exit_code': process.returncode}
    except subprocess.TimeoutExpired:
        installation = {'status': 'timeout', 'limit_seconds': 420}
(WORK / 'installation.json').write_text(json.dumps(installation, indent=2))
print(installation)
print((WORK / 'install.log').read_text(errors='replace')[-3000:])

In [ ]:
(WORK / 'kaggle_preflight.py').write_text("\"\"\"Bounded E4B runtime probe v4: chat-template quality check + tile-16 kernel numerics.\n\nv3 established that a source-level `_get_tile_size` guard (tile 16 when head_size > 128)\nlets the quantized E4B checkpoint generate on a Tesla T4, but the output was degenerate\nand repetitive. v4 answers two questions and nothing else:\n\n1. Kernel numerics: does the tile-16 Triton attention kernel agree with a reference\n   scaled-dot-product implementation for the checkpoint's head dimensions (256/512)?\n2. Chat template: does applying the model's chat template (absent in v1-v3, which fed\n   raw strings) change the degenerate behaviour?\n\nNo agent runs; no official scoring; one bounded subprocess.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport importlib\nimport json\nimport math\nimport os\nimport platform\nimport sys\nimport time\nfrom pathlib import Path\n\nMODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'\n\n\ndef model_config(model_path: Path) -> dict:\n    model_path = Path(model_path)\n    if not (model_path / 'config.json').is_file():\n        raise FileNotFoundError(f'Missing model config: {model_path / \"config.json\"}')\n    return {\n        'model': str(model_path), 'dtype': 'half',\n        'tensor_parallel_size': 1, 'max_model_len': 2048,\n        'gpu_memory_utilization': 0.80, 'enforce_eager': True,\n        'disable_log_stats': True,\n        'limit_mm_per_prompt': {'image': 0, 'video': 0, 'audio': 0},\n    }\n\n\ndef compatibility_gate(gpus: list[dict]) -> dict:\n    if not gpus:\n        return {'status': 'no_cuda_gpu', 'attention_risk': None}\n    turing = any(tuple(gpu['capability']) < (8, 0) for gpu in gpus)\n    return {\n        'status': 'requires_runtime_test', 'attention_risk': turing,\n        'note': 'Memory capacity alone does not establish attention-kernel support.',\n    }\n\n\ndef _candidate_module_paths():\n    \"\"\"Locate triton_unified_attention.py without importing the module.\"\"\"\n    import glob as _glob\n    paths = []\n    try:\n        import importlib.util\n        spec = importlib.util.find_spec('vllm.v1.attention.ops.triton_unified_attention')\n        if spec and spec.origin:\n            paths.append(Path(spec.origin))\n    except Exception:\n        pass\n    for pattern in ('/usr/local/lib/python*/dist-packages/vllm/v1/attention/ops/triton_unified_attention.py',\n                    '/usr/local/lib/python*/site-packages/vllm/v1/attention/ops/triton_unified_attention.py'):\n        paths.extend(Path(p) for p in _glob.glob(pattern))\n    seen, unique = set(), []\n    for path in paths:\n        key = str(path)\n        if key not in seen and path.is_file():\n            seen.add(key)\n            unique.append(path)\n    return unique\n\n\nANCHOR = ('    # Default behavior\\n'\n          '    if is_prefill:\\n'\n          '        return 32\\n')\nGUARD = ('    # SM75_TILE_GUARD: Turing cannot allocate 3*tile*head*2 bytes of shared\\n'\n         '    # memory for 512-wide heads (vLLM issue #38918). Fall back to tile 16.\\n'\n         '    if head_size > 128:\\n'\n         '        return 16\\n')\n\n\ndef apply_sm75_tile_patch(extra_paths=None) -> dict:\n    \"\"\"Rewrite the installed vLLM source so spawned engine processes inherit the guard.\"\"\"\n    info = {'source_files': [], 'capability_major': None, 'applied_modules': [], 'verify': None}\n    try:\n        import torch\n        if not torch.cuda.is_available():\n            info['capability_major'] = 'no_cuda'\n            return info\n        major = torch.cuda.get_device_capability()[0]\n        info['capability_major'] = major\n        if major >= 8:\n            info['skipped'] = 'capability >= 8.0'\n            return info\n    except Exception as error:\n        info['capability_major'] = 'error: ' + str(error)\n        return info\n\n    paths = list(extra_paths or []) + _candidate_module_paths()\n    for path in paths:\n        try:\n            text = path.read_text(encoding='utf-8')\n        except OSError as error:\n            info['source_files'].append({'path': str(path), 'status': 'unreadable: ' + str(error)})\n            continue\n        if 'SM75_TILE_GUARD' in text:\n            info['source_files'].append({'path': str(path), 'status': 'already_patched'})\n            continue\n        if ANCHOR not in text:\n            info['source_files'].append({'path': str(path), 'status': 'anchor_not_found'})\n            continue\n        patched = text.replace(ANCHOR, GUARD + ANCHOR, 1)\n        try:\n            path.write_text(patched, encoding='utf-8')\n        except OSError as error:\n            info['source_files'].append({'path': str(path), 'status': 'unwritable: ' + str(error)})\n            continue\n        import hashlib\n        info['source_files'].append({\n            'path': str(path), 'status': 'patched',\n            'sha256_before': hashlib.sha256(text.encode('utf-8')).hexdigest(),\n            'sha256_after': hashlib.sha256(patched.encode('utf-8')).hexdigest(),\n        })\n\n    # Also patch any already-imported copy so the parent process stays consistent.\n    import importlib\n    for name in ('vllm.v1.attention.ops.triton_unified_attention',\n                 'vllm.attention.ops.triton_unified_attention',\n                 'vllm.v1.attention.ops.triton_unified_attention_diffkv'):\n        try:\n            module = importlib.import_module(name)\n            original = module._get_tile_size\n        except Exception:\n            continue\n        def patched_fn(*args, _original=original, **kwargs):\n            if args and isinstance(args[0], int) and args[0] > 128:\n                return 16\n            return _original(*args, **kwargs)\n        module._get_tile_size = patched_fn\n        info['applied_modules'].append(name)\n    if info['applied_modules']:\n        try:\n            first = importlib.import_module(info['applied_modules'][0])\n            info['verify'] = {'head512_decode': first._get_tile_size(512, 0, 2, False),\n                              'head512_prefill': first._get_tile_size(512, 0, 2, True)}\n        except Exception as error:\n            info['verify'] = 'error: ' + str(error)\n    return info\n\n\ndef save(result: dict, output_path: Path) -> None:\n    output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')\n\n\ndef numeric_check(result: dict, output_path: Path) -> None:\n    \"\"\"Compare the tile-16 Triton kernel against a float32 SDPA reference.\"\"\"\n    import torch\n    import torch.nn.functional as F\n    stage = {'cases': [], 'status': 'failed'}\n    result['numeric_check'] = stage\n    save(result, output_path)\n    try:\n        from vllm.v1.attention.ops.triton_unified_attention import unified_attention\n        torch.manual_seed(0)\n        device = 'cuda'\n        stage['tile_prefill_head512'] = __import__(\n            'vllm.v1.attention.ops.triton_unified_attention',\n            fromlist=['_get_tile_size'])._get_tile_size(512, 0, 2, True)\n        for head_dim, num_heads, kv_len, decode in (\n            (512, 4, 128, False),\n            (512, 4, 1, True),\n            (256, 4, 128, False),\n            (256, 4, 1, True),\n        ):\n            block_size = 16\n            q_len = 1 if decode else kv_len\n            num_blocks = (kv_len + block_size - 1) // block_size\n            q = torch.randn(q_len, num_heads, head_dim, dtype=torch.float16, device=device)\n            k_cache = torch.randn(num_blocks, block_size, num_heads, head_dim,\n                                  dtype=torch.float16, device=device)\n            v_cache = torch.randn_like(k_cache)\n            out = torch.empty_like(q)\n            block_table = torch.arange(num_blocks, dtype=torch.int32, device=device).view(1, -1)\n            seqused_k = torch.tensor([kv_len], dtype=torch.int32, device=device)\n            cu = torch.tensor([0, q_len], dtype=torch.int32, device=device)\n            unified_attention(q, k_cache, v_cache, out, cu, q_len, seqused_k, kv_len,\n                              1.0 / math.sqrt(head_dim), True, (-1, -1), block_table,\n                              0.0, None, None, None)\n            k_seq = k_cache.reshape(-1, num_heads, head_dim)[:kv_len]\n            v_seq = v_cache.reshape(-1, num_heads, head_dim)[:kv_len]\n            ref = F.scaled_dot_product_attention(\n                q.transpose(0, 1).unsqueeze(0).float(),\n                k_seq.transpose(0, 1).unsqueeze(0).float(),\n                v_seq.transpose(0, 1).unsqueeze(0).float(),\n                is_causal=not decode)\n            ref = ref.squeeze(0).transpose(0, 1)\n            triton_err = (out.float() - ref).abs().max().item()\n            sdpa16 = F.scaled_dot_product_attention(\n                q.transpose(0, 1).unsqueeze(0),\n                k_seq.transpose(0, 1).unsqueeze(0),\n                v_seq.transpose(0, 1).unsqueeze(0),\n                is_causal=not decode).squeeze(0).transpose(0, 1)\n            sdpa_err = (sdpa16.float() - ref).abs().max().item()\n            rel = triton_err / ref.abs().max().item()\n            stage['cases'].append({\n                'head_dim': head_dim, 'kv_len': kv_len, 'decode': decode,\n                'triton_max_abs_err_fp32ref': round(triton_err, 6),\n                'sdpa_fp16_max_abs_err_fp32ref': round(sdpa_err, 6),\n                'triton_rel_err': round(rel, 6),\n            })\n        worst = max(c['triton_rel_err'] for c in stage['cases'])\n        stage['worst_rel_err'] = worst\n        stage['status'] = 'pass' if worst < 0.05 else 'fail'\n    except Exception as error:\n        stage['error_type'] = type(error).__name__\n        stage['error'] = str(error)[:800]\n    save(result, output_path)\n\n\ndef manual_gemma(text: str) -> str:\n    return f'<start_of_turn>user\\n{text}<end_of_turn>\\n<start_of_turn>model\\n'\n\n\ndef extract_text(chat_result):\n    if isinstance(chat_result, list):\n        return chat_result[0].outputs[0].text\n    return chat_result.outputs[0].text\n\n\ndef chat_probe(result: dict, llm, output_path: Path) -> None:\n    from vllm import SamplingParams\n    stage = {'prompts': [], 'status': 'failed'}\n    result['chat_probe'] = stage\n    save(result, output_path)\n    prompts = [\n        ('sum', 'Write a Python function add(a, b) that returns their sum.'),\n        ('math', 'What is 17 * 24? Reply with only the number.'),\n        ('unit_test', 'Explain in one sentence what a unit test is.'),\n    ]\n    try:\n        for name, text in prompts:\n            entry = {'name': name, 'prompt': text}\n            raw = llm.generate([text], SamplingParams(max_tokens=32, temperature=0))\n            entry['raw_32tok'] = raw[0].outputs[0].text[:300]\n            try:\n                chat = llm.chat([{'role': 'user', 'content': text}],\n                                SamplingParams(max_tokens=64, temperature=0))\n                entry['chat_path'] = 'tokenizer_chat_template'\n            except Exception as error:\n                entry['chat_path'] = 'manual_gemma_format (template error: ' + str(error)[:100] + ')'\n                chat = llm.generate([manual_gemma(text)],\n                                    SamplingParams(max_tokens=64, temperature=0))\n            entry['chat_64tok'] = extract_text(chat)[:300]\n            stage['prompts'].append(entry)\n            save(result, output_path)\n        stage['status'] = 'pass'\n    except Exception as error:\n        stage['error_type'] = type(error).__name__\n        stage['error'] = str(error)[:800]\n    save(result, output_path)\n\n\ndef run_preflight(model_path: Path | None, *, generate: bool, output_path: Path) -> dict:\n    started = time.monotonic()\n    result = {\n        'schema_version': 4, 'model': MODEL_NAME,\n        'python': platform.python_version(), 'platform': platform.platform(),\n        'generation_status': 'not_requested', 'agent_status': 'not_run',\n        'configuration_scope': 'short-context runtime probe, not official scoring',\n        'packages': {}, 'gpus': [],\n    }\n    for package in ('torch', 'vllm', 'transformers', 'compressed-tensors', 'swegemma', 'adk-submission'):\n        try:\n            result['packages'][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            result['packages'][package] = None\n    try:\n        import torch\n        if torch.cuda.is_available():\n            result['gpus'] = [\n                {'name': torch.cuda.get_device_name(i),\n                 'capability': list(torch.cuda.get_device_capability(i)),\n                 'total_memory_bytes': torch.cuda.get_device_properties(i).total_memory}\n                for i in range(torch.cuda.device_count())\n            ]\n    except ImportError:\n        pass\n    result['compatibility'] = compatibility_gate(result['gpus'])\n    save(result, output_path)\n    if not generate:\n        result['elapsed_seconds'] = round(time.monotonic() - started, 3)\n        return result\n    result['generation_status'] = 'failed'\n    try:\n        if not result['gpus']:\n            raise RuntimeError('No CUDA GPU is available; no model loading attempted.')\n        if model_path is None:\n            raise FileNotFoundError('Provide an attached E4B checkpoint path.')\n        os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')\n        os.environ.setdefault('VLLM_NO_USAGE_STATS', '1')\n        os.environ.setdefault('HF_HUB_OFFLINE', '1')\n        os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')\n        result['sm75_tile_patch'] = apply_sm75_tile_patch()\n        save(result, output_path)\n        numeric_check(result, output_path)\n        if result['numeric_check']['status'] != 'pass':\n            raise RuntimeError(\n                'Tile-16 kernel numerics failed; skipping model load. '\n                + json.dumps(result['numeric_check'])[:400])\n        from vllm import LLM, SamplingParams\n        checkpoint = json.loads((Path(model_path) / 'config.json').read_text(encoding='utf-8'))\n        text = checkpoint.get('text_config', checkpoint)\n        result['checkpoint_attention'] = {k: text.get(k) for k in ('head_dim', 'global_head_dim', 'dtype', 'torch_dtype')}\n        load_started = time.monotonic()\n        llm = LLM(**model_config(model_path))\n        result['load_seconds'] = round(time.monotonic() - load_started, 3)\n        result['generation_status'] = 'passed'\n        result['agent_status'] = 'not_run'\n        save(result, output_path)\n        chat_probe(result, llm, output_path)\n    except Exception as error:\n        result['error_type'] = type(error).__name__\n        result['error'] = str(error)[:1200]\n    result['elapsed_seconds'] = round(time.monotonic() - started, 3)\n    save(result, output_path)\n    return result\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--model-path', type=Path)\n    parser.add_argument('--generate', action='store_true')\n    parser.add_argument('--output', type=Path, required=True)\n    args = parser.parse_args(argv)\n    result = run_preflight(args.model_path, generate=args.generate, output_path=args.output)\n    print(json.dumps(result, ensure_ascii=False, indent=2))\n    return int(args.generate and result.get('chat_probe', {}).get('status') != 'pass')\n\n\nif __name__ == '__main__':\n    sys.exit(main())\n", encoding='utf-8')
MODEL = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2')
if installation['status'] != 'passed':
    result = {'generation_status': 'not_run', 'agent_status': 'not_run', 'reason': 'runtime_installation_failed'}
else:
    env = os.environ.copy()
    env.update(CUDA_VISIBLE_DEVICES='0', VLLM_WORKER_MULTIPROC_METHOD='spawn', TRANSFORMERS_NO_TF='1', TRANSFORMERS_NO_FLAX='1', VLLM_NO_USAGE_STATS='1', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
    with (WORK / 'generation.log').open('w') as log:
        try:
            process = subprocess.run([sys.executable, str(WORK / 'kaggle_preflight.py'), '--model-path', str(MODEL), '--generate', '--output', str(WORK / 'model_preflight.json')], env=env, stdout=log, stderr=subprocess.STDOUT, timeout=600, start_new_session=True)
            if (WORK / 'model_preflight.json').exists():
                result = json.loads((WORK / 'model_preflight.json').read_text())
            else:
                result = {'generation_status': 'failed', 'agent_status': 'not_run', 'exit_code': process.returncode, 'reason': 'worker_exited_without_result'}
        except subprocess.TimeoutExpired:
            result = {'generation_status': 'timeout', 'agent_status': 'not_run', 'limit_seconds': 600}
    print((WORK / 'generation.log').read_text(errors='replace')[-5000:])
result['notebook_elapsed_seconds'] = round(time.monotonic() - START, 3)
(WORK / 'model_preflight.json').write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))

The JSON and logs in notebook outputs are the evidence. `numeric_check` reports the
tile used and worst relative error against the float32 reference; `chat_probe`
records raw vs templated completions for three prompts. A passing numeric check with
coherent templated output permits a subsequent single-task agent smoke experiment;
anything else closes this line without silently changing configuration.